In [60]:
def get_style(dark=True):
    # Color Palettes
    bg = "#0f172a" if dark else "#f8fafc"
    text = "#f8fafc" if dark else "#1e293b"
    card = "#1e293b" if dark else "#ffffff"
    border = "#334155" if dark else "#e2e8f0"
    
    return HTML(f"""
    <style>
        /* Force the background on every possible container layer */
        html, body, #jp-main-content-panel, .jp-Notebook, .v-application, 
        .voila-app, .output_wrapper, .widget-subarea {{ 
            background-color: {bg} !important; 
            color: {text} !important;
        }}

        .widget-label {{ min-width: 160px !important; color: {text} !important; }}
        
        .summary-box {{
            background-color: {card}; 
            border-left: 5px solid #22c55e;
            border: 1px solid {border};
            padding: 15px; margin-bottom: 20px; border-radius: 8px;
            color: {text} !important; font-family: sans-serif;
        }}
        
        .dataframe {{ 
            background-color: {card} !important; color: {text} !important; 
            width: 100%; border-collapse: collapse; 
        }}
        .dataframe th {{ 
            background-color: {border} !important; color: {text} !important; 
            text-align: left !important; padding: 12px 8px !important; 
        }}
        .dataframe td {{ 
            border-bottom: 1px solid {border} !important; 
            padding: 10px 8px !important; color: {text} !important; 
        }}
        
        .jupyter-button {{ min-width: 150px !important; height: 35px !important; }}
        .download-btn {{ 
            cursor:pointer; background-color:{card}; border:1px solid {border}; 
            padding:6px 12px; border-radius:6px; font-size: 13px; color: {text};
        }}
    </style>
    """)

In [61]:
# 1. THEME TOGGLE (Top Right)
theme_toggle = widgets.ToggleButton(
    value=True, 
    description='🌙 Dark Mode', 
    button_style='info', 
    icon='moon',
    layout=widgets.Layout(width='150px')
)

# 2. INPUT WIDGETS
title_input = widgets.Text(description="Bill Name:", placeholder="e.g. Electric Bill")
comment_input = widgets.Textarea(
    description="Comments:", 
    placeholder="Optional details...", 
    layout=widgets.Layout(height='60px')
)
andrew_input = widgets.FloatText(description="Andrew Paid ($):", value=0.0)
moi_input = widgets.FloatText(description="Moi Paid ($):", value=0.0)

# 3. ACTION BUTTONS
calc_button = widgets.Button(description="Calculate & Save", button_style='success')
delete_button = widgets.Button(description="Delete Last", button_style='danger')

# 4. MODAL POPUP ELEMENTS (For Deletion)
confirm_yes = widgets.Button(description="Yes, Delete It", button_style='danger')
confirm_no = widgets.Button(description="Cancel")

# 5. PLACEHOLDERS (Where the magic happens)
summary_output = widgets.Output()
download_output = widgets.Output()
output_area = widgets.Output()
popup_output = widgets.Output()

# 6. LAYOUT ASSEMBLY
# Pack the buttons together
button_row = widgets.HBox([calc_button, delete_button, download_output])

# Assemble the final page
layout = widgets.VBox([
    widgets.HBox([
        widgets.HTML("<h2>⚖️ Bill Balancer</h2>"), 
        theme_toggle
    ], layout=widgets.Layout(justify_content='space-between')),
    summary_output, 
    title_input, 
    comment_input, 
    andrew_input, 
    moi_input, 
    widgets.HTML("<br>"),
    button_row, 
    output_area,
    popup_output
])

In [62]:
# --- 1. THEME SWAPPER LOGIC ---
def on_theme_change(change):
    """Updates the CSS variables when the toggle is clicked."""
    with style_output:
        clear_output()
        display(get_style(dark=change['new']))
    if change['new']:
        theme_toggle.description = '🌙 Dark Mode'
    else:
        theme_toggle.description = '☀️ Light Mode'

# --- 2. DATA & UI FUNCTIONS ---
def display_summary():
    with summary_output:
        clear_output()
        if not history_df.empty:
            current_month = datetime.now().strftime("%Y-%m")
            temp_df = history_df.copy()
            temp_df['Total_Num'] = temp_df['Total'].replace(r'[\$,]', '', regex=True).astype(float)
            monthly_data = temp_df[temp_df['Date'].str.startswith(current_month)]
            monthly_total = monthly_data['Total_Num'].sum()
            display(HTML(f'<div class="summary-box"><strong>📊 {datetime.now().strftime("%B %Y")} Summary</strong><br>Total Spent: ${monthly_total:.2f} across {len(monthly_data)} entries.</div>'))

def update_history_table():
    with output_area:
        clear_output()
        if not history_df.empty:
            display(HTML("<h3>📝 Recent History</h3>"))
            # Hides row numbers and flips order so newest is on top
            display(HTML(history_df.tail(10)[::-1].to_html(index=False, classes='dataframe')))

def create_download_link():
    if os.path.exists(FILE):
        df = pd.read_csv(FILE)
        csv_data = df.to_csv(index=False)
        b64 = base64.b64encode(csv_data.encode()).decode()
        filename = f"Bill_Balancer_{datetime.now().strftime('%Y_%m_%d')}.csv"
        return HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}" style="text-decoration:none;"><button class="download-btn">📥 Download for Excel</button></a>')
    return HTML("")

# --- 3. ACTION HANDLERS ---
def on_click(b):
    global history_df
    total = andrew_input.value + moi_input.value
    share = total / 2
    res = f"Moi owes Andrew ${andrew_input.value-share:.2f}" if andrew_input.value > share else f"Andrew owes Moi ${moi_input.value-share:.2f}" if moi_input.value > share else "Even split!"
    
    new_row = {"Date": datetime.now().strftime("%Y-%m-%d"), "Bill": title_input.value, "Total": f"${total:.2f}", "Result": res, "Comments": comment_input.value}
    history_df = pd.concat([history_df, pd.DataFrame([new_row])], ignore_index=True)
    history_df.to_csv(FILE, index=False)
    
    title_input.value = ""; comment_input.value = ""; andrew_input.value = 0.0; moi_input.value = 0.0
    refresh_ui()

def show_delete_popup(b):
    with popup_output:
        clear_output()
        display(HTML("""
            <div style="position: fixed; top: 0; left: 0; width: 100%; height: 100%; background: rgba(0,0,0,0.6); z-index: 9999; display: flex; justify-content: center; align-items: center;">
                <div style="background: white; padding: 30px; border-radius: 12px; text-align: center; border: 2px solid #ef4444; color: #333;">
                    <h3>⚠️ Are you sure?</h3>
                    <p>This will permanently remove the last entry.</p>
                </div>
            </div>
        """))
        display(widgets.HBox([confirm_yes, confirm_no], layout={'justify_content':'center'}))

def close_popup(b):
    popup_output.clear_output()

def final_delete(b):
    global history_df
    if not history_df.empty:
        history_df = history_df.drop(history_df.index[-1])
        history_df.to_csv(FILE, index=False)
        refresh_ui()
    close_popup(None)

def refresh_ui():
    display_summary()
    update_history_table()
    with download_output:
        clear_output(); display(create_download_link())

# --- 4. THE "SPARK" (INITIALIZATION) ---
theme_toggle.observe(on_theme_change, names='value')
calc_button.on_click(on_click)
delete_button.on_click(show_delete_popup)
confirm_yes.on_click(final_delete)
confirm_no.on_click(close_popup)

# Set the starting theme and display the app
with style_output:
    display(get_style(dark=True))

display(style_output, layout)
refresh_ui()

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<IPython.core.display.HTML object>', '…